In [ ]:
import rasterio
import numpy as np
import glob
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# =========================
# 1. 文件路径
# =========================
folder = str(config.FEATURES)

rf_files = sorted(glob.glob(folder + r"\*_RF.tif"))
ca_files = sorted(glob.glob(folder + r"\*_CA.tif"))
all_files = rf_files + ca_files

print("RF:", len(rf_files), "CA:", len(ca_files))

# =========================
# 2. reference grid（统一栅格）
# =========================
ref_path = rf_files[0]

with rasterio.open(ref_path) as ref:
    ref_shape = (ref.height, ref.width)
    profile = ref.profile.copy()
    h, w = ref_shape

# =========================
# 3. 栅格对齐函数（不依赖CRS）
# =========================
def align_no_crs(f):

    with rasterio.open(f) as src:
        data = src.read(1).astype(np.float32)

        out = np.full(ref_shape, np.nan, dtype=np.float32)

        hh = min(h, data.shape[0])
        ww = min(w, data.shape[1])

        out[:hh, :ww] = data[:hh, :ww]

    return out

# =========================
# 4. 构建特征矩阵 X
# =========================
features = []

for f in all_files:
    arr = align_no_crs(f)
    features.append(arr.flatten())

X = np.column_stack(features)

print("X shape:", X.shape)

# =========================
# 5. 读取矿点CSV → labels（全部当矿点）
# =========================
mine_df = pd.read_csv(
    config.SAMPLE_PUL,
    header=None,
    names=["x", "y", "label"]
)


labels = np.zeros(X.shape[0], dtype=int)

with rasterio.open(ref_path) as src:

    width = src.width
    height = src.height

    hit = 0

    for _, row in mine_df.iterrows():

        x = float(row["x"])
        y = float(row["y"])

        try:
            r, c = src.index(x, y)

            if 0 <= r < height and 0 <= c < width:

                idx = r * width + c
                labels[idx] = 1
                hit += 1

            else:
                print("❌越界:", x, y, r, c)

        except Exception as e:
            print("❌失败:", x, y, e)

print("成功写入矿点数:", hit)
print("labels sum:", labels.sum())

# =========================
# 6. mask（去掉NaN像元）
# =========================
mask = ~np.isnan(X).any(axis=1)

X_clean = X[mask]
labels_clean = labels[mask]

P_index = np.where(labels_clean == 1)[0]
U_index = np.where(labels_clean == 0)[0]

print("P:", len(P_index), "U:", len(U_index))
print("labels sum (before mask):", np.sum(labels))
print("labels_clean sum:", np.sum(labels_clean))
print("P_index:", len(P_index))
# =========================
# 7. PU Bagging RF
# =========================
T = 100
nP = len(P_index)

pred_matrix = np.zeros((X_clean.shape[0], T))

for t in range(T):

    # 防止样本不足
    sizeU = min(nP, len(U_index))
    sampled_U = np.random.choice(U_index, size=sizeU, replace=False)

    train_idx = np.concatenate([P_index, sampled_U])

    X_train = X_clean[train_idx]
    y_train = np.concatenate([
        np.ones(len(P_index)),
        np.zeros(len(sampled_U))
    ])

    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        random_state=t,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    pred_matrix[:, t] = rf.predict_proba(X_clean)[:, 1]

# =========================
# 8. PU概率 + 不确定性
# =========================
pu_prob = pred_matrix.mean(axis=1)
pu_std = pred_matrix.std(axis=1)

print("PU完成")

# =========================
# 9. 回填栅格
# =========================
full_prob = np.full(X.shape[0], np.nan)
full_std = np.full(X.shape[0], np.nan)

full_prob[mask] = pu_prob
full_std[mask] = pu_std

pu_map = full_prob.reshape(h, w)
std_map = full_std.reshape(h, w)

# =========================
# 9.5 填补空洞
# =========================
valid_mask = ~np.isnan(pu_map)

pu_map[~valid_mask] = np.interp(
    np.flatnonzero(~valid_mask),
    np.flatnonzero(valid_mask),
    pu_map[valid_mask]
)

valid_mask_std = ~np.isnan(std_map)

std_map[~valid_mask_std] = np.interp(
    np.flatnonzero(~valid_mask_std),
    np.flatnonzero(valid_mask_std),
    std_map[valid_mask_std]
)
# =========================
# 10. 输出 GeoTIFF
# =========================
profile.update(
    dtype="float32",
    count=1,
    nodata=-9999,
    compress="lzw"
)

out_dir = config.ensure_dir(config.OUTPUT / "pu_ca_rf")
out_prob = str(out_dir / "PU_CA_RF_result.tif")
out_std = str(out_dir / "PU_CA_RF_uncertainty.tif")

with rasterio.open(out_prob, "w", **profile) as dst:
    tmp = pu_map.copy()
    tmp[np.isnan(tmp)] = -9999
    dst.write(tmp.astype(np.float32), 1)

with rasterio.open(out_std, "w", **profile) as dst:
    tmp = std_map.copy()
    tmp[np.isnan(tmp)] = -9999
    dst.write(tmp.astype(np.float32), 1)

print("输出完成：PU概率图 + 不确定性图")